# Module 02 — Clean SEC Bond Data

This module parses nested fields (`maturity`, `coupon`) and builds analysis-ready date/rate columns.

In [1]:
from pathlib import Path
import ast
import re
import pandas as pd
import numpy as np

In [2]:
candidate_paths = [
    Path('/Users/game./Documents/Sandbox/sec_api_data.csv'),
    Path('/Users/game./Documents/Sandbox/Bond_starter_project_public/data/processed/sec_bond_raw_snapshot.csv'),
    Path('/Users/game./Documents/Sandbox/bond_features_by_company.csv'),
]

source_path = next((p for p in candidate_paths if p.exists()), None)
if source_path is None:
    raise FileNotFoundError('No source file found for cleaning module.')

df = pd.read_csv(source_path, low_memory=False)
print(f'Loaded: {source_path}')
print(f'Rows: {len(df):,} | Columns: {df.shape[1]:,}')

Loaded: /Users/game./Documents/Sandbox/sec_api_data.csv
Rows: 221,709 | Columns: 23


In [3]:
def parse_payload(value):
    if isinstance(value, dict):
        return value
    if pd.isna(value):
        return {}
    if isinstance(value, str):
        text = value.strip()
        if text.startswith('{') and text.endswith('}'):
            try:
                return ast.literal_eval(text)
            except Exception:
                return {}
    return {}

date_col = 'maturity__issue_date' if 'maturity__issue_date' in df.columns else None
if date_col is None and 'maturity' in df.columns:
    maturity_obj = df['maturity'].apply(parse_payload)
    df['maturity__issue_date'] = maturity_obj.apply(lambda x: x.get('issue_date'))
    date_col = 'maturity__issue_date'

if date_col is not None:
    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')

rate_col = 'coupon__rate' if 'coupon__rate' in df.columns else None
if rate_col is None and 'coupon' in df.columns:
    coupon_obj = df['coupon'].apply(parse_payload)
    df['coupon__rate_raw'] = coupon_obj.apply(lambda x: x.get('rate'))

    def extract_rate(v):
        if pd.isna(v):
            return np.nan
        m = re.search(r'(\d+(?:\.\d+)?)', str(v))
        return float(m.group(1)) if m else np.nan

    df['coupon__rate'] = df['coupon__rate_raw'].apply(extract_rate)
    rate_col = 'coupon__rate'

if rate_col is not None:
    df[rate_col] = pd.to_numeric(df[rate_col], errors='coerce')

before = len(df)
df = df.dropna(subset=[c for c in [date_col, rate_col] if c is not None]).drop_duplicates()
after = len(df)

print(f'Using date column: {date_col}')
print(f'Using rate column: {rate_col}')
print(f'Rows before cleaning: {before:,}')
print(f'Rows after cleaning:  {after:,}')

Using date column: maturity__issue_date
Using rate column: coupon__rate
Rows before cleaning: 221,709
Rows after cleaning:  214,205


In [4]:
out_dir = Path('/Users/game./Documents/Sandbox/Bond_starter_project_public/data/processed')
out_dir.mkdir(parents=True, exist_ok=True)
cleaned_path = out_dir / 'sec_bond_cleaned.csv'
df.to_csv(cleaned_path, index=False)
print(f'Saved cleaned dataset: {cleaned_path}')

Saved cleaned dataset: /Users/game./Documents/Sandbox/Bond_starter_project_public/data/processed/sec_bond_cleaned.csv
